In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import os
from natsort import natsorted

import scanpy as sc
import seaborn as sns

from scroutines import basicu

import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from statsmodels.tools.sm_exceptions import ValueWarning
from tqdm import tqdm


import sys
sys.path.insert(0, '/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/myvisctx/analysis_multiome/')
import lmm

In [2]:
outfigdir = '/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/'
f = '/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/data/v1_multiome/superdupermegaRNA_hasraw_cheng22_astro_v2.h5ad'
adata = sc.read(f)
adata

AnnData object with n_obs × n_vars = 7697 × 15573
    obs: 'Age', 'Doublet', 'Doublet Score', 'n_counts', 'n_genes', 'percent_mito', 'sample', 'Type', 'Subclass', 'Class', 'Sample', 'total_counts', 'pct_counts_mt', 'n_genes_by_counts', 'total_counts_mt', 'Doublet?', 'Study', 'Type_leiden', 'Time', 'Light', 'curated_cluster', 'pc1', 'pc2'
    var: 'feature_types'
    uns: 'leiden', 'ngbr_astro'
    obsm: 'pc_astro'
    layers: 'norm'
    obsp: 'ngbr_astro_connectivities', 'ngbr_astro_distances'

In [3]:
adata.obs['curated_cluster']

AAAGGGCCATCGAAGG-1-P28_1a-1       1
AACAACCTCTAACGCA-1-P28_1a-1       1
AACAAGATCTAATTCC-1-P28_1a-1       1
AACACACCAGACCTAT-1-P28_1a-1       1
AACAGGGGTTGTGGCC-1-P28_1a-1       1
                                 ..
ATCGATGCATGCCATA-1-P38_dr_2b-5    2
TCTACCGCATCGGCCA-1-P38_dr_2b-5    2
GGGAAGTTCCATGATG-1-P38_dr_2b-5    2
AAGCGTTGTGTTGACT-1-P38_dr_2b-5    1
TTGTGGAAGCTCTATG-1-P38_dr_2b-5    2
Name: curated_cluster, Length: 7697, dtype: category
Categories (4, object): ['0', '1', '2', '3']

In [4]:
%%time

obs_fixed1 = 'Time'
obs_fixed2 = 'Light'
obs_random = 'Sample'

offset = 1e-2
scale = 1e4

for cluster in ['0', '1', '3', '2']:
    tag = f'd260129_c{cluster}'
    output = os.path.join(outfigdir, f'NRDR_DEGs_LMM_astro_cheng22_{tag}.h5ad')


    adatasub = adata[adata.obs['curated_cluster']==f'{cluster}']

    # ### test
    # adatasub = adatasub[:,:20]
    # ### test

    genes = adatasub.var.index.values 

    obs = adatasub.obs[[obs_fixed1, obs_fixed2, obs_random]].copy()
    obs = obs.dropna()

    adatasub = adatasub[obs.index]

    # mat (CP10k norm)
    mat = np.array(adatasub.X.todense())/adatasub.obs['n_counts'].values.reshape(-1,1)*scale

    res = lmm.run_lmm_two_fixed(mat, genes, obs, obs_fixed1, obs_fixed2, obs_random, output=output, offset=offset)
    print(output)

(2767, 15573) (2767, 3)
(2767, 15563) (2767, 3)
(2767, 12114) (2767, 3)


  0% 2/12114 [00:00<1:33:22,  2.16it/s]


LinAlgError: Singular matrix